# ARC Prize 2026 ARC-AGI-2: Qwen3.8-Flash-Next agent on 4 L4 GPUs

The method and the measurements are in the [writeup](https://github.com/itsuitsuki/arc-prize-2026-arc-agi-2/blob/main/WRITEUP.md).

# Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

STARTED = time.time()
SCORING = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))  # True when Kaggle runs the notebook to score it, False when it is saved
CHALLENGES = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
WHEELS = Path("/kaggle/input/datasets/nisshokuitsuki/arc-merlin-wheels")
FILES = Path("/kaggle/input/datasets/nisshokuitsuki/arc-solution-files")
INTEL = Path("/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1")
MTP = Path("/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1")
NGRAM_TABLE = Path("/kaggle/input/models/nisshokuitsuki/arc-fp8-ngram-table/transformers/default/1")
AZ_FOCUS = Path("/kaggle/input/models/nisshokuitsuki/arc-az-focus-override/transformers/default/1/override.safetensors")
for path in (CHALLENGES, WHEELS, FILES, INTEL, MTP, NGRAM_TABLE, AZ_FOCUS):
    assert path.exists(), path
TASKS = json.loads(CHALLENGES.read_text())
Path("/kaggle/tmp").mkdir(exist_ok=True)
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2",
                  OPENBLAS_NUM_THREADS="1", VLLM_USE_DEEP_GEMM="0")
gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
assert len(gpus) == 4 and all("L4" in name for name in gpus), gpus

## Engine

The wheels go into their own virtual environment, and the notebook imports its packages from there too.

In [ ]:
VENV = Path("/kaggle/tmp/merlin-venv")
PYTHON = str(VENV / "bin/python")
if SCORING:
    subprocess.check_call([sys.executable, "-m", "venv", "--without-pip", "--clear", str(VENV)])
    uv_wheel, uv = next(WHEELS.glob("uv-*.whl")), Path("/kaggle/tmp/uv")
    with zipfile.ZipFile(uv_wheel) as archive:
        uv.write_bytes(archive.read(next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))))
    uv.chmod(0o755)
    wheels = Path("/kaggle/tmp/wheels")
    shutil.rmtree(wheels, ignore_errors=True)
    wheels.mkdir()
    for wheel in WHEELS.glob("*.whl"):
        if wheel != uv_wheel:  # Kaggle drops the '+' of the engine's local version from its file name
            (wheels / wheel.name.replace("0.30.0merlin", "0.30.0+merlin")).symlink_to(wheel)
    subprocess.check_call([str(uv), "pip", "install", "--python", PYTHON, "--no-index", "--no-deps", *sorted(map(str, wheels.iterdir()))],
                          env={**os.environ, "UV_CACHE_DIR": "/kaggle/tmp/uv-cache", "UV_LINK_MODE": "copy", "UV_NO_PROGRESS": "1"})
    SITE = Path(subprocess.check_output([PYTHON, "-c", 'import sysconfig; print(sysconfig.get_paths()["purelib"])'], text=True).strip())
    for module in WHEELS.glob("flashinfer-aot-sm89-*.so"):  # FlashInfer's sampling and norm modules, compiled ahead of time for sm89
        name = module.stem.removeprefix("flashinfer-aot-sm89-")
        (SITE / "flashinfer/data/aot" / name).mkdir(parents=True, exist_ok=True)
        shutil.copy(module, SITE / "flashinfer/data/aot" / name / f"{name}.so")
    loaded = [name for name in ("torch", "transformers", "tokenizers", "jinja2", "httpx", "huggingface_hub") if name in sys.modules]
    assert not loaded, f"already imported from the image: {loaded}"
    sys.path.insert(0, str(SITE))

## Engine patches

`patch()` adds a file of `engine_patches/` to the installed vLLM, so that every engine process runs it. The writeup explains the patches.

In [ ]:
import ast


def patch(name: str, source: str) -> None:
    compile(source, name, "exec")
    init = SITE / "vllm/__init__.py"
    lines = init.read_text().split("\n")
    entry = f"_arc_module({name!r}, {source!r})"
    if entry in lines:
        return
    added = [i for i, line in enumerate(lines) if line.startswith("_arc_module(")]
    if added:
        lines.insert(added[-1] + 1, entry)  # after the earlier patches, so that they run in the order of the calls
    else:
        head = 0  # after the module docstring and any `from __future__` import, which must come first
        for node in ast.parse("\n".join(lines)).body:
            docstring = isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str)
            if not (docstring or (isinstance(node, ast.ImportFrom) and node.module == "__future__")):
                break
            head = node.end_lineno
        lines[head:head] = ["import sys as _arc_sys", "import types as _arc_types", "",
                            "def _arc_module(name, source):",
                            "    module = _arc_types.ModuleType(name)",
                            "    _arc_sys.modules[name] = module",
                            "    exec(compile(source, f'<{name}>', 'exec'), module.__dict__)", "", entry]
    text = "\n".join(lines)
    init.write_text(text)
    compile(text, str(init), "exec")

In [ ]:
ENGINE_PATCHES = [  # files of engine_patches/, applied in this order
    "arc_patches",  # chain(), with which the patches below hook engine modules
    "arc_ple_flag",  # each GPU worker profiles its memory after the n-gram table is loaded
    "arc_weight_override",  # AZ-Focus weights over Intel's checkpoint
    "arc_reap",  # REAP: drop the pruned experts while loading
    "arc_read_filter",  # each worker reads only the tensors it keeps
    "arc_draft_vocab",  # FR-Spec: the drafter proposes tokens from our vocabulary only
    "arc_topk_logits",  # top-k 20 logits gathered across tensor parallelism
    "arc_qsa_host_kv",  # K/V of the sparse-attention layers in host memory
    "arc_late_commit",  # the second stage commits the GDN state after sending the drafts
    "arc_gdn_reuse",  # one GDN metadata build per step for all GDN groups
    "arc_last_state",  # keep the newest GDN state at a block boundary across turns
    "arc_gdn_park",  # park idle GDN states in host memory
    "arc_verify_chunk",  # verify draft tokens in 32 MiB chunks
]
if SCORING:
    for name in ENGINE_PATCHES:
        patch(name, (FILES / "engine_patches" / f"{name}.py").read_text())

## Model

In [ ]:
import struct
from contextlib import nullcontext
from typing import BinaryIO

def safetensors_header(source: Path | BinaryIO) -> tuple[dict, int]:
    with (open(source, "rb") if isinstance(source, (str, Path)) else nullcontext(source)) as stream:
        size = struct.unpack("<Q", stream.read(8))[0]
        if not 0 < size < 100_000_000:
            raise ValueError(f"{source}: invalid safetensors header length {size}")
        entries = json.loads(stream.read(size))
    entries.pop("__metadata__", None)
    return entries, 8 + size


MTP_SHARD = "model_extra_tensors.safetensors"
FP8_TABLE_GLOB = "model-plefp8-*.safetensors"
DRAFT_QUANTIZATION = {
    "config_groups": {"mtp_routed_experts": {
        "format": "pack-quantized", "input_activations": None, "output_activations": None, "targets": [r"re:.*\.mlp\.experts\..*"],
        "weights": {"actorder": None, "block_structure": None, "dynamic": False, "group_size": 32, "num_bits": 4, "observer": "minmax",
                    "observer_kwargs": {}, "strategy": "group", "symmetric": True, "type": "int"}}},
    "format": "pack-quantized", "global_compression_ratio": None, "ignore": [r"re:^(?!.*\.mlp\.experts\.).*$"],
    "kv_cache_scheme": None, "quant_method": "compressed-tensors", "quantization_status": "compressed", "sparsity_config": {},
    "transform_config": {}}


def link_others(source: Path, output: Path, skip: set[str]) -> None:
    for entry in source.iterdir():
        if entry.name not in skip and not entry.name.startswith(".") and not entry.name.endswith(".safetensors") \
                and not (output / entry.name).exists():
            (output / entry.name).symlink_to(entry)


def assemble_model(intel: Path, fp8_table: Path, output: Path, num_experts: int) -> Path:
    weight_map = {name: file for name, file in json.loads((intel / "model.safetensors.index.json").read_text())["weight_map"].items()
                  if file != MTP_SHARD}
    tables = sorted(fp8_table.glob(FP8_TABLE_GLOB))
    if not tables:
        raise FileNotFoundError(f"no {FP8_TABLE_GLOB} in {fp8_table}")
    for table in tables:
        weight_map.update({name: table.name for name in safetensors_header(table)[0]})
    output.mkdir(parents=True, exist_ok=False)
    for file in sorted(set(weight_map.values())):
        (output / file).symlink_to((fp8_table if file.startswith("model-plefp8-") else intel) / file)
    config = json.loads((intel / "config.json").read_text())
    text = config.get("text_config", config)
    text["ple_embedding_dtype"] = "float8_e4m3fn"
    text["num_experts"] = num_experts
    (output / "config.json").write_text(json.dumps(config, indent=2))
    (output / "model.safetensors.index.json").write_text(json.dumps({"metadata": {}, "weight_map": weight_map}))
    link_others(intel, output, {"config.json", "model.safetensors.index.json"})
    return output


def assemble_drafter(drafter: Path, output: Path, num_experts: int) -> Path:
    output.mkdir(parents=True, exist_ok=False)
    for entry in drafter.iterdir():
        if entry.name != "config.json":
            (output / entry.name).symlink_to(entry)
    config = json.loads((drafter / "config.json").read_text())
    config["quantization_config"] = DRAFT_QUANTIZATION
    config.get("text_config", config)["num_experts"] = num_experts
    (output / "config.json").write_text(json.dumps(config, indent=2))
    return output


if SCORING:
    counts = {len(kept) for kept in json.loads((FILES / "reap448_keep.json").read_text()).values()}
    assert len(counts) == 1, counts  # every layer keeps as many experts
    experts = counts.pop()
    shutil.rmtree("/kaggle/tmp/flashnext", ignore_errors=True)
    MODEL = assemble_model(INTEL, NGRAM_TABLE, Path("/kaggle/tmp/flashnext/model"), experts)
    DRAFTER = assemble_drafter(MTP, Path("/kaggle/tmp/flashnext/drafter"), experts)

## Page-cache prefetch

In [ ]:
import contextlib
from concurrent.futures import ThreadPoolExecutor


def read_file(path: Path) -> None:
    with contextlib.suppress(OSError), open(path, "rb", buffering=0) as handle:
        while handle.read(16 << 20):
            pass


def prefetch(model: Path) -> None:  # reads the shards once, so that the server loads them from the page cache
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(read_file, sorted(path for path in model.glob("*.safetensors") if "plefp8" not in path.name)))


if SCORING:
    threading.Thread(target=prefetch, args=(MODEL,), daemon=True).start()

# Running

## Task prompt and answer parsing

In [ ]:
import re
from typing import Any

import numpy as np

def normalize_grid(value: Any) -> list[list[int]] | None:
    try:
        array = np.asarray(value)
        if array.ndim != 2 or array.size == 0 or any(not 0 < size <= 30 for size in array.shape):
            return None
        if not np.issubdtype(array.dtype, np.integer):
            if not np.all(np.equal(array, np.floor(array))):
                return None
        array = array.astype(int)
        if array.min() < 0 or array.max() > 9:
            return None
        return array.tolist()
    except (TypeError, ValueError, OverflowError):
        return None


def is_valid_grid(value: Any) -> bool:
    return normalize_grid(value) is not None


# prompts/direct.txt and prompts/task.txt follow arcprize/arc-agi-benchmarking (MIT License, Copyright (c) 2025 ARC Prize Foundation); see THIRD_PARTY_NOTICES.md and licenses/arc-agi-benchmarking-MIT.md.
DIRECT_PROMPT = (FILES / "prompts/direct.txt").read_text().strip()
TASK_PROMPT = (FILES / "prompts/task.txt").read_text().strip()


def format_grid(grid: list) -> str:
    return json.dumps(grid)


def training_examples(task: dict) -> str:
    return "".join(f"--Example {i}-- \n\n INPUT: \n\n{format_grid(pair['input'])}\n\nOUTPUT: \n\n{format_grid(pair['output'])}\n\n"
                   for i, pair in enumerate(task["train"]))


def task_prompt(task: dict) -> str:
    if len(task["test"]) == 1:
        return DIRECT_PROMPT.format(training_examples=training_examples(task), test_input=format_grid(task["test"][0]["input"]))
    tests = "\n\n".join(f"--Test Input {k}--\n{format_grid(test['input'])}\n--End of Test Input {k}--"
                         for k, test in enumerate(task["test"], 1))
    return TASK_PROMPT.format(training_examples=training_examples(task), test_inputs=tests)


# The answer parsing follows arcprize/arc-agi-benchmarking (MIT License, Copyright (c) 2025 ARC Prize Foundation); see THIRD_PARTY_NOTICES.md and licenses/arc-agi-benchmarking-MIT.md.
TEST_OUTPUT_LABEL = r"--\s*Test Output\s*(\d+)\s*--"


def json_lists(text: str) -> list:
    found, depth, start = [], 0, 0
    for position, char in enumerate(text):
        if char == "[":
            start = position if depth == 0 else start
            depth += 1
        elif char == "]" and depth:
            depth -= 1
            if depth == 0:
                with contextlib.suppress(ValueError, RecursionError):
                    found.append(json.loads(text[start:position + 1]))
    return found


def all_grids(text: str) -> list:
    return [grid for grid in json_lists(text) if is_valid_grid(grid)]


def answer_grids(text: str, count: int) -> list:
    parts = re.split(TEST_OUTPUT_LABEL, text)
    found = {int(label): last_grid(body) for label, body in zip(parts[1::2], parts[2::2])}
    if found:
        return [found.get(k) for k in range(1, count + 1)]
    grids = all_grids(text)[-count:] if count else []
    return grids + [None] * (count - len(grids))


def last_grid(text: str) -> list | None:
    candidates = [match.group(1) for match in re.finditer(r"\\boxed\{(.*?)\}", text, re.DOTALL)][:1]
    end = text.rfind("]")
    if end >= 0:
        depth = 0
        for start in range(end, -1, -1):
            depth += {"]": 1, "[": -1}.get(text[start], 0)
            if depth == 0:
                candidates.append(text[start:end + 1])
                break
    for candidate in candidates:
        try:
            grid = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if is_valid_grid(grid):
            return grid
    return None

## System prompt and tools

In [ ]:
TOOLS = json.loads((FILES / "tools.json").read_text())
SYSTEM_PROMPT = (FILES / "prompts/system.txt").read_text().strip()
AVOID_PREVIOUS = "\n\n" + (FILES / "prompts/avoid_previous.txt").read_text().strip()
PREVIOUS_PROGRAM = "\n\n" + (FILES / "prompts/previous_program.txt").read_text().strip()
PREVIOUS_CODE_LIMIT = 12000


def previous_program_text(program: dict, grids: list) -> str:
    passed, total = program["train_passed"], program["train_total"]
    conclusion = ("Its program passed every training pair, which does not prove it right on the test input." if passed == total
                  else f"Its program passed {passed} of {total} training pairs.")
    code = program["code"] if len(program["code"]) <= PREVIOUS_CODE_LIMIT else program["code"][:PREVIOUS_CODE_LIMIT] + "\n# ... (cut)"
    return PREVIOUS_PROGRAM.format(conclusion=conclusion, code=code, report="\n".join(program["report"]),
                                   answer=previous_answer_text(grids))


def previous_answer_text(grids: list) -> str:
    shown = [format_grid(grid) if grid is not None else "(no grid)" for grid in grids]
    return shown[0] if len(shown) == 1 else "\n\n".join(f"--Test Output {k}--\n{grid}" for k, grid in enumerate(shown, 1))


def initial_messages(task: dict, previous: list | None = None, program: dict | None = None,
                     ndimage: bool = False) -> list[dict]:
    system = SYSTEM_PROMPT
    if ndimage:
        system = system.replace("libraries: numpy, ", "libraries: numpy, scipy.ndimage (predefined as ndimage, or from scipy import ndimage), ", 1)
    content = task_prompt(task) + ("" if previous is None else AVOID_PREVIOUS.format(answer=previous_answer_text(previous))
                                   if program is None else previous_program_text(program, previous))
    return [{"role": "system", "content": system}, {"role": "user", "content": content}]

## Model input and output

In [ ]:
def decode_tokens(tokenizer, ids):
    return tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)


def load_tokenizer(model):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(str(model), local_files_only=True, trust_remote_code=False)
    terminals = {tokenizer.eos_token_id}
    for name in ("config.json", "generation_config.json"):
        path = Path(model) / name
        if path.exists():
            config = json.loads(path.read_text())
            value = config.get("eos_token_id", config.get("text_config", {}).get("eos_token_id"))
            terminals.update(value if isinstance(value, list) else [value])
    terminals.discard(None)
    tokenizer.terminal_ids = sorted(terminals)
    return tokenizer


def render_messages(tokenizer, messages: list[dict], tools: list[dict]) -> tuple[str, list[int]]:
    template = tokenizer.get_chat_template()
    marker = "{%- if loop.index0 > ns.last_query_index %}"
    if template.count(marker) == 1:
        template = template.replace(marker, "{%- if true %}")
    render = lambda conversation: tokenizer.apply_chat_template(
        conversation, tools=tools, chat_template=template, tokenize=False, add_generation_prompt=True,
        enable_thinking=True, preserve_thinking=True, truncate_history_thinking=False,
    )
    probe = [{"role": "user", "content": "a"},
             {"role": "assistant", "content": "b", "reasoning_content": "EARLIER-REASONING-PROBE"},
             {"role": "user", "content": "c"}]
    if "EARLIER-REASONING-PROBE" not in render(probe):
        raise ValueError("Unsupported native reasoning template")
    return encode_prompt(tokenizer, render(messages), "<|im_start|>assistant\n<think>\n")


def encode_prompt(tokenizer, text: str, generation_prompt: str) -> tuple[str, list[int]]:
    ids = tokenizer.encode(text, add_special_tokens=False)
    if decode_tokens(tokenizer, ids) != text or not text.endswith(generation_prompt):
        raise ValueError("Native tool prompt does not round-trip")
    return text, ids


REPEATED_THINK_END = re.compile(r"</think>(?:\s*</think>)+")


def collapse_think_end(body: str) -> str:
    return REPEATED_THINK_END.sub("</think>", body, count=1)


def parse_assistant(body: str) -> dict:
    body = collapse_think_end(body)
    if not body.startswith("<think>") or body.count("</think>") != 1:
        raise ValueError("Missing complete reasoning boundary")
    reasoning, _, content = body[len("<think>"):].partition("</think>")
    content = content.strip()
    message: dict[str, Any] = {"role": "assistant", "reasoning_content": reasoning.strip(), "content": content}
    if "<tool_call>" not in content:
        return message
    prefix, _, _ = content.partition("<tool_call>")
    blocks = re.findall(r"<tool_call>\s*<function=([^>]+)>\s*(.*?)\s*</function>\s*</tool_call>", content, re.DOTALL)
    remainder = re.sub(r"<tool_call>\s*<function=[^>]+>\s*.*?\s*</function>\s*</tool_call>", "", content[len(prefix):], flags=re.DOTALL)
    if not blocks or remainder.strip() or len(blocks) > 4:
        raise ValueError("Malformed or excessive tool calls")
    calls = []
    for name, arguments in blocks:
        pairs = re.findall(r"<parameter=(\w+)>\s*\n?(.*?)\n?\s*</parameter>", arguments, re.DOTALL)
        values = {key: value.strip() for key, value in pairs}
        # the exact text between the template's own "<parameter=x>\n" and "\n</parameter>", so that the next turn's rendering of this message repeats the generated tokens; the tools take the stripped values (tool_arguments)
        exact_pairs = re.findall(r"<parameter=(\w+)>\n(.*?)\n</parameter>", arguments, re.DOTALL)
        kept = dict(exact_pairs) if [key for key, _ in exact_pairs] == [key for key, _ in pairs] else values
        required, allowed = ("outputs", ()) if name == "arc_submit_answer" else ("code", ("target",))
        if ((required == "code" and name != "arc_python_executor") or required not in values or len(values) != len(pairs)
                or set(values) - {required, *allowed}
                or re.sub(r"<parameter=\w+>.*?</parameter>", "", arguments, flags=re.DOTALL).strip()):
            raise ValueError("Unsupported tool name or parameters")
        calls.append({"type": "function", "function": {"name": name, "arguments": kept}})
    message.update(content=prefix.strip(), tool_calls=calls)
    return message


def tool_arguments(call: dict) -> dict:
    return {key: value.strip() for key, value in call["function"]["arguments"].items()}


def validate_response(tokenizer, ids: list[int], response: dict, max_tokens: int, thinking_token_budget: int | None) -> str | None:
    if len(response["choices"]) != 1:
        raise ValueError("Expected one native completion")
    choice = response["choices"][0]
    if choice.get("finish_reason") == "abort":  # the server ended the request
        raise ValueError("Native request aborted by the server")
    generated = choice.get("token_ids")
    if (choice.get("prompt_token_ids") != ids or not isinstance(generated, list)
            or len(generated) != response["usage"]["completion_tokens"]
            or response["usage"]["prompt_tokens"] != len(ids)
            or not 0 < len(generated) <= max_tokens
            or decode_tokens(tokenizer, generated) != choice["text"]):
        raise ValueError("Native response token accounting differs")
    end_ids = tokenizer.encode("</think>", add_special_tokens=False)
    body = decode_tokens(tokenizer, generated[:-1])
    has_thinking_end = len(end_ids) == 1 and end_ids[0] in generated
    if (choice["finish_reason"] == "length" and thinking_token_budget is not None and not has_thinking_end
            and len(generated) == max_tokens <= thinking_token_budget):
        return None  # the turn's token allowance ran out inside the reasoning, before the thinking budget did
    if thinking_token_budget is not None:
        if len(end_ids) != 1:
            raise ValueError("Native thinking token budget was not enforced")
        if has_thinking_end and generated.index(end_ids[0]) >= max(1, thinking_token_budget):
            raise ValueError("Native thinking token budget was not enforced")
    if choice["finish_reason"] == "length":
        return None
    terminal_ids = tokenizer.terminal_ids
    if (choice["finish_reason"] != "stop" or generated[-1] not in terminal_ids
            or any(t in terminal_ids for t in generated[:-1])):
        raise ValueError("Native message terminator differs")
    if has_thinking_end:
        return "<think>\n" + body
    if "<tool_call>" in body:
        reasoning, marker, content = body.partition("<tool_call>")
        return f"<think>\n{reasoning.rstrip()}\n</think>\n{marker}{content}"
    return "<think>\n</think>\n" + body


def reasoning_tokens(tokenizer, response: dict) -> int:
    generated = response["choices"][0]["token_ids"]
    end = tokenizer.encode("</think>", add_special_tokens=False)
    return generated.index(end[0]) if len(end) == 1 and end[0] in generated else len(generated)

## Sandbox

`sandbox.py` checks the model's programs and runs them in a separate process with limits on CPU time, files and output.

In [ ]:
sys.path.insert(0, str(FILES))
import sandbox

sandbox.PYTHON = PYTHON
from sandbox import STDOUT_LIMIT, grid_problem, ndimage_error, run_session, safety_error

## Tools

In [ ]:
from collections import Counter

def grid_difference(expected: list, actual, error: str | None) -> str:
    if actual == expected:
        return "passed."
    if actual is None:
        return f"failed: no valid grid ({error or 'invalid output'})."
    shape = lambda grid: f"{len(grid)}x{len(grid[0])}"
    if shape(actual) != shape(expected):
        return f"failed: expected a {shape(expected)} grid, got {shape(actual)}:\n{format_grid(actual)}"
    cells = [(r, c, e, a) for r, (want, got) in enumerate(zip(expected, actual))
             for c, (e, a) in enumerate(zip(want, got)) if e != a]
    shown = ", ".join(f"({r},{c}) expected {e} got {a}" for r, c, e, a in cells[:20])
    return f"failed: {len(cells)} of {sum(map(len, expected))} cells differ: {shown}{', ...' if len(cells) > 20 else ''}."


def verification_lines(task: dict, record: dict, outputs: list, errors: list, target: str) -> list[str]:
    count = len(task["train"])
    record.update(train_outputs=outputs[:count], test_outputs=outputs[count:],
                  train_passed=sum(actual == pair["output"] for actual, pair in zip(outputs, task["train"])))
    lines = [f"Training pair {number}: {grid_difference(pair['output'], actual, error)}"
             for number, (pair, actual, error) in enumerate(zip(task["train"], outputs, errors), 1)]
    lines.append(f"{record['train_passed']}/{count} training pairs passed.")
    for number, (grid, error) in enumerate(zip(outputs[count:], errors[count:]) if target == "test" else [], 1):
        lines += [f"Test input {number} output:", format_grid(grid) if grid is not None else f"no valid grid ({error or 'invalid output'})"]
    return lines


def printed_lines(result: dict) -> list[str]:
    return (["Output:", result["stdout"].rstrip("\n")] if result["stdout"] else []) + (
        [f"... (output truncated at {STDOUT_LIMIT} characters)"] if result["truncated"] else []) + (
        [f"Error: {result['error']}"] if result["error"] else [])


def session_entry(record: dict) -> tuple[str | None, dict]:
    return record["code"] if record.get("ok") else record.get("kept"), record.get("functions") or {}


def kept_statements(code: str, completed: int) -> str:
    end = ast.parse(code).body[completed - 1]
    data = code.encode()
    lines = data.splitlines(keepends=True)
    return data[:sum(map(len, lines[:end.end_lineno - 1])) + end.end_col_offset].decode()


# Invariants that the training outputs of a task may all keep of their inputs. 
# A test output that breaks a kept one gets a note in the tool reply, and the scheduler counts its attempt as unverified. 
# The background of a grid is its most common color.
INVARIANTS = {"colors": "uses exactly its input's colors",
              "background": "keeps every background cell of its input at the input's shape",
              "count": "has as many non-background cells as its input"}


def broken_invariants(task: dict, predictions) -> list[tuple[int, str, str]]:
    def colors(grid):
        return {cell for row in grid for cell in row}

    def background(grid):
        return Counter(cell for row in grid for cell in row).most_common(1)[0][0]

    def non_background(grid):
        own = background(grid)
        return sum(cell != own for row in grid for cell in row)

    def colors_rule(source, output):
        added, dropped = sorted(colors(output) - colors(source)), sorted(colors(source) - colors(output))
        parts = [f"adds color{'s' * (len(added) > 1)} {', '.join(map(str, added))}"] * bool(added)
        parts += [f"drops color{'s' * (len(dropped) > 1)} {', '.join(map(str, dropped))}"] * bool(dropped)
        return " and ".join(parts) or None

    def background_rule(source, output):
        if len(source) != len(output) or any(len(a) != len(b) for a, b in zip(source, output)):
            return "has another shape than its input"
        own = background(source)
        changed = sum(o != s for a, b in zip(source, output) for s, o in zip(a, b) if s == own)
        return f"changes {changed} of its input's background cells" if changed else None

    def count_rule(source, output):
        have, want = non_background(output), non_background(source)
        return f"has {have} non-background cells where its input has {want}" if have != want else None

    rules = {"colors": colors_rule, "background": background_rule, "count": count_rule}
    try:
        kept = [key for key, rule in rules.items() if all(rule(pair["input"], pair["output"]) is None for pair in task["train"])]
        broken = []
        for number, (grid, test) in enumerate(zip(predictions or [], task["test"]), 1):
            if isinstance(grid, list) and grid and all(isinstance(row, list) and row for row in grid):
                broken += [(number, key, detail) for key in kept if (detail := rules[key](test["input"], grid)) is not None]
        return broken
    except (IndexError, TypeError):
        return []


def invariant_hints(task: dict, outputs: list) -> list[str]:
    return [f"Note: every training output {INVARIANTS[key]}, but this output for test input {number} {detail}. "
            "Check whether that is intended before you submit." for number, key, detail in broken_invariants(task, outputs)]


def run_python_tool(arguments: dict, task: dict, timeout: float, session: list, ndimage: bool = False) -> tuple[dict, str]:
    code, target = arguments["code"], arguments.get("target", "train")
    count = len(task["train"])
    record = {"code": code, "target": target, "error": None, "train_passed": 0, "train_total": count,
              "train_outputs": None, "test_outputs": None}
    if target not in ("train", "test"):
        return {**record, "error": "unknown target"}, 'The target must be "train" or "test".'
    error = safety_error(code, ndimage)
    if error:
        return {**record, "error": error}, f"The program was rejected by the static gate: {error}."
    result = run_session(code, task["train"], [{"input": test["input"]} for test in task["test"]], timeout, session,
                         ndimage=ndimage)
    if result is None:
        return {**record, "error": "timeout or worker failure", "ok": False}, "The program timed out or the worker process failed."
    record.update(stdout=result["stdout"], script_error=result["error"], ok=result["error"] is None, functions={})
    if result["error"] and result["completed"]:
        record["kept"] = kept_statements(code, result["completed"])
    lines, best = printed_lines(result), -1
    for name, run in result["functions"].items():
        checked: dict = {}
        passed = all(actual == pair["output"] for actual, pair in zip(run["outputs"], task["train"]))
        lines += [f"{name}:", *verification_lines(task, checked, run["outputs"], run["errors"], "test" if passed else target)]
        if passed and all(grid is not None for grid in checked["test_outputs"]):
            lines += invariant_hints(task, checked["test_outputs"])
        record["functions"][name] = {"train_passed": checked["train_passed"], "train_total": count,
                                     "test_outputs": checked["test_outputs"]}
        rank = 2 if passed and all(grid is not None for grid in checked["test_outputs"]) else int(passed)
        if rank >= best:
            record.update(checked)
            best = rank
    return record, "\n".join(lines) if lines else "The program ran and printed nothing."


SUBMIT_KINDS = ("function", "variable", "grid")


def submit_answer(outputs: str, task: dict, timeout: float, session: list, pending=None,
                  final: bool = False, previous: list | None = None, ndimage: bool = False) -> tuple[dict, str | None]:
    record = {"outputs": outputs, "error": None, "grids": None, "sources": None, "confirm": False}

    def refuse(problem: str) -> tuple[dict, str]:
        return {**record, "error": problem}, f"The answer was not submitted: {problem}. Fix it and call arc_submit_answer again."

    count = len(task["test"])
    try:
        value = json.loads(outputs)
    except (ValueError, RecursionError) as error:  # deep nesting exceeds the decoder's recursion limit
        return refuse(f'outputs is not valid JSON ({error}); give e.g. [{{"function": "transform"}}]')
    entries = [value] * count if isinstance(value, dict) and ("function" in value or count == 1) else value
    if not isinstance(entries, list) or len(entries) != count:
        return refuse(f"outputs needs one entry per test input, in order ({count})")
    problems, grids, named = [], [None] * count, []
    for index, entry in enumerate(entries):
        kind = next(iter(entry)) if isinstance(entry, dict) and len(entry) == 1 else None
        if kind not in SUBMIT_KINDS:
            problems.append(f"output {index + 1}: give exactly one of {', '.join(SUBMIT_KINDS)}")
        elif kind == "grid":
            grids[index] = normalize_grid(entry["grid"])
            if grids[index] is None:
                problems.append(f"output {index + 1}: the grid is not valid: {grid_problem(entry['grid']) or 'not a list of rows of integers 0-9'}")
        elif not (isinstance(entry[kind], str) and entry[kind].isidentifier() and not entry[kind].startswith("_")):
            problems.append(f"output {index + 1}: {kind} must be the name of a session {kind}")
        else:
            named.append([index, kind, entry[kind]])
    if problems:
        return refuse("; ".join(problems))
    failing = {}
    if named:
        result = run_session("", task["train"], [{"input": test["input"]} for test in task["test"]], timeout, session,
                             submit=named, ndimage=ndimage)
        if result is None:
            return refuse("evaluating it timed out or the worker process failed")
        problems = [f"output {index + 1}: {entry['problem']}" for (index, _, _), entry in zip(named, result["entries"])
                    if entry["problem"]]
        if problems:
            return refuse("; ".join(problems))
        for (index, _, _), entry in zip(named, result["entries"]):
            grids[index] = entry["grid"]
        failing = {name: pairs for name, pairs in result["failing"].items() if pairs}
    if failing and not final and value != pending:
        lines = [f"{name} fails training pair{'s' if len(pairs) > 1 else ''} {', '.join(map(str, pairs))} of "
                 f"{len(task['train'])}." for name, pairs in failing.items()]
        return {**record, "confirm": True}, "\n".join(lines + [
            "To submit it anyway, call arc_submit_answer again with identical arguments; otherwise fix it first."])
    if previous is not None and grids == [normalize_grid(grid) for grid in previous] and not final and value != pending:
        return {**record, "confirm": True}, ("These outputs are the previous attempt's answer, and your final answer must differ "
                                             "from it. To submit it anyway, call arc_submit_answer again with identical "
                                             "arguments; otherwise change it first.")
    sources = ["grid" if kind == "grid" else f"{kind}:{name}" for kind, name in (next(iter(e.items())) for e in entries)]
    return {**record, "grids": grids, "sources": sources, "failing": failing}, None

## Agent loop

In [ ]:
import builtins
from threading import Lock


def complete(http, url: str, tokenizer, payload: dict, timeout: float) -> dict:
    model, prompt, max_tokens, budget = payload["model"], payload["prompt"], payload["max_tokens"], payload["thinking_token_budget"]

    def generate(ids, limit):
        if limit <= 0:
            return [], "length"
        body = {"model": model, "prompt": ids, "max_tokens": limit, "temperature": payload["temperature"], "top_p": payload["top_p"],
                "seed": payload["seed"], "top_k": payload["top_k"], "stop_token_ids": payload["stop_token_ids"],
                "return_token_ids": True, "add_special_tokens": False, "skip_special_tokens": False,
                "include_stop_str_in_output": True, "spaces_between_special_tokens": False}
        response = http.post(url, json=body, timeout=timeout)
        response.raise_for_status()
        out = response.json()
        choice = out["choices"][0]
        if out["model"] != model or out["usage"]["prompt_tokens"] != len(ids) or choice.get("prompt_token_ids") != ids:
            raise ValueError("vLLM model or prompt tokens differ")
        return list(choice["token_ids"]), choice["finish_reason"]

    # once the reasoning reaches the thinking budget, </think> is appended and generation goes on for the answer
    think_start, think_end = tokenizer.convert_tokens_to_ids("<think>"), tokenizer.convert_tokens_to_ids("</think>")
    opened = prompt[::-1].index(think_start) if think_start in prompt else None  # tokens after the last <think>
    first = max_tokens if budget is None or opened is None else min(max_tokens, max(0, budget - opened))
    tokens, finish = generate(prompt, first)
    if finish == "length" and len(tokens) < max_tokens:
        forced = [] if think_end in tokens else [think_end]
        rest, finish = generate(prompt + tokens + forced, max_tokens - len(tokens) - len(forced))
        tokens = tokens + forced + rest
    return {"model": model, "choices": [{"text": decode_tokens(tokenizer, tokens), "token_ids": tokens, "prompt_token_ids": prompt,
                                         "finish_reason": finish}],
            "usage": {"prompt_tokens": len(prompt), "completion_tokens": len(tokens)}}


RESULT_LOG_LOCK = Lock()


def write_json(path: Path, value) -> None:
    with path.open("x") as stream:
        json.dump(value, stream, indent=2)


class TokenBudgetSpent(Exception):
    pass


def request_payload(tokenizer, messages: list[dict], model: str, settings: dict, seed: int, prefill: str | None = None,
                    continuation: tuple[str, list[int]] | None = None, cap: int | None = None) -> dict:
    text, ids = render_messages(tokenizer, messages, TOOLS)
    if continuation is not None:
        # the server must see the tokens it sampled: an encoding of the same text can split them otherwise and fork the cached prefix
        previous_text, previous_ids = continuation
        if text.startswith(previous_text):
            joined = previous_ids + tokenizer.encode(text[len(previous_text):], add_special_tokens=False)
            if decode_tokens(tokenizer, joined) == text:
                ids = joined
    if prefill:
        text, ids = text + prefill, ids + tokenizer.encode(prefill, add_special_tokens=False)
    budget = None if prefill else max(0, min(settings["thinking_token_budget"],
                                             settings["max_model_len"] - len(ids) - answer_room(settings)))
    max_tokens = min(settings["max_tokens"], settings["max_model_len"] - len(ids))
    if max_tokens <= 0:
        raise TokenBudgetSpent
    if cap is not None:
        max_tokens = min(max_tokens, max(cap, answer_room(settings)))
        if budget is not None:
            budget = max(0, min(budget, max_tokens - answer_room(settings)))
    return {"model": model, "prompt": ids, "temperature": settings["temperature"], "top_p": settings["top_p"], "top_k": settings["top_k"],
            "max_tokens": max_tokens, "thinking_token_budget": budget, "seed": seed, "stop_token_ids": tokenizer.terminal_ids}


FINAL_ANSWER = {"role": "user", "content": "The tool budget is used up: do not call the tool again. Give your final "
                                            "answer now, the output grid for each test input in the required format."}
FINAL_SUBMIT = {"role": "user", "content": "The tool budget is used up: do not call arc_python_executor again. Give "
                                               "your final answer now: call arc_submit_answer with the function or "
                                               "variable that computed it, or write the output grid for each test input "
                                               "in the required format."}
FINAL_ANSWER_RESERVE = 2048
THINK_CLOSED = "<think>\n</think>\n"
THINK_END = "\n</think>\n\n"


def final_prefill(task: dict) -> str:
    return THINK_END + ("--Test Output 1--\n" if len(task["test"]) > 1 else "") + "[["


def absorb_prefill(body, prefill: str):
    if body is None or not body.startswith(THINK_CLOSED):
        return body
    return THINK_CLOSED + prefill.split("</think>\n\n", 1)[1] + body[len(THINK_CLOSED):]


def answer_room(settings: dict) -> int:
    return settings["max_tokens"] - settings["thinking_token_budget"]


def context_tokens(tokenizer, messages: list[dict]) -> int:
    return len(render_messages(tokenizer, messages, TOOLS)[1])


def final_turn(settings: dict, turn_index: int, context: int) -> bool:
    return (turn_index == settings["max_turns"] - 1
            or settings["max_model_len"] - context < answer_room(settings) + FINAL_ANSWER_RESERVE)


def turn_clamp(settings: dict, now: float) -> int | None:
    at, rate = settings.get("turn_clamp_at"), settings.get("turn_clamp_rate")
    return None if at is None or not rate else max(0, int((at - now) * rate))


def start_turn(settings: dict, task: dict, tokenizer, messages: list[dict], turn_index: int, turns: list[dict],
                force: bool = False) -> tuple[bool, str | None]:
    passed = last_passing(turns) is not None
    if not (force or final_turn(settings, turn_index, context_tokens(tokenizer, messages))):
        return False, None
    messages.append(FINAL_SUBMIT if passed else FINAL_ANSWER)
    return True, THINK_END if passed else final_prefill(task)


def passing_outputs(calls: list[dict]) -> list | None:
    for call in reversed(calls):
        outputs = call.get("test_outputs")
        if call.get("train_passed") == call.get("train_total") and outputs and all(grid is not None for grid in outputs):
            return list(outputs)
    return None


def last_passing(turns: list[dict]) -> list | None:
    return passing_outputs([call for turn in turns for call in turn.get("calls") or []])


def program_fallback(turns: list[dict]) -> list | None:
    for turn in reversed(turns):
        if any(grid is not None for grid in turn.get("prediction") or []):
            return turn["prediction"]
        outputs = passing_outputs(turn.get("calls") or [])
        if outputs is not None:
            return outputs
    return None


def program_predictions(turns: list[dict], prediction) -> list | None:
    outputs = last_passing(turns)
    return outputs if outputs is not None and outputs != prediction else None


def verified(turns: list[dict], prediction) -> bool:
    if not prediction or any(grid is None for grid in prediction):
        return False
    runs = [run for turn in turns for call in turn.get("calls") or [] for run in [call, *(call.get("functions") or {}).values()]]
    passing = [run["test_outputs"] for run in runs
               if run.get("train_total") and run.get("train_passed") == run["train_total"] and run.get("test_outputs")]
    return all(any(i < len(outputs) and outputs[i] == grid for outputs in passing) for i, grid in enumerate(prediction))


SESSION_NAMES = {"train", "test", "checks", "objects", "diff"}


def free_names(tree) -> set[str]:
    bound, loaded = set(), set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Name):
            (loaded if isinstance(node.ctx, ast.Load) else bound).add(node.id)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            bound.add(node.name)
        elif isinstance(node, ast.arg):
            bound.add(node.arg)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            bound.update((alias.asname or alias.name).split(".")[0] for alias in node.names)
    return loaded - bound - set(dir(builtins)) - SESSION_NAMES


def defining_statements(code: str, earlier: list[str]) -> str:
    try:
        needed = free_names(ast.parse(code))
    except SyntaxError:
        return ""
    found = {}  # name -> (call index, line, source)
    while needed:
        name = needed.pop()
        for index in range(len(earlier) - 1, -1, -1):
            try:
                tree = ast.parse(earlier[index])
            except SyntaxError:
                continue
            hit = None
            for node in tree.body:
                names = ({node.name} if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) else
                         {(a.asname or a.name).split(".")[0] for a in node.names} if isinstance(node, (ast.Import, ast.ImportFrom)) else
                         {t.id for t in getattr(node, "targets", [getattr(node, "target", None)]) if isinstance(t, ast.Name)}
                         if isinstance(node, (ast.Assign, ast.AnnAssign)) else set())
                if name in names:
                    hit = node
            if hit is not None:
                key = (index, hit.lineno)
                if key not in {v[:2] for v in found.values()}:
                    found[name] = (index, hit.lineno, ast.get_source_segment(earlier[index], hit))
                    needed |= free_names(hit) - set(found)
                break
    parts = sorted({v for v in found.values() if v[2]})
    return "".join(f"{source}\n" for _, _, source in parts)


def previous_program(directory: Path, task: dict) -> dict | None:
    turns = [json.loads(path.read_text()) for path in sorted(directory.glob("[0-9][0-9]_execution.json"))]
    calls = [call for turn in turns for call in turn.get("calls") or [] if call.get("code")]
    if not calls:
        return None
    sources = next((turn["sources"] for turn in reversed(turns) if turn.get("sources")), None) or []
    names = {source.split(":", 1)[1] for source in sources if source.startswith("function:")}
    call = (next((c for c in reversed(calls) if names & set(c.get("functions") or {})), None)
            or next((c for c in reversed(calls) if c.get("ok")), calls[-1]))
    outputs = call.get("train_outputs") or [None] * len(task["train"])
    error = call.get("error") or call.get("script_error")
    report = [f"Training pair {number}: {grid_difference(pair['output'], actual, error)}"
              for number, (pair, actual) in enumerate(zip(task["train"], outputs), 1)]
    code = call["code"]
    prefix = defining_statements(code, [c["code"] for c in calls[:calls.index(call)] if c.get("ok")])
    code = f"# defined in earlier calls\n{prefix}\n{code}" if prefix else code
    return {"code": code, "train_passed": call.get("train_passed") or 0, "train_total": len(task["train"]),
            "report": report}


def answer_retry(answer: list, problems: list[str]) -> dict:
    count, read = len(answer), sum(grid is not None for grid in answer)
    labels = " Put \"--Test Output k--\" on the line before the k-th grid." if count > 1 else ""
    head = (f"Your final answer must give {count} output grid{'s' if count > 1 else ''}, each as a JSON list of rows of "
            f"integers 0-9 such as [[0, 1], [2, 3]]; {read} of {count} could be read")
    return {"role": "user", "content": f"{head} ({'; '.join(problems)}).{labels} If a function or variable computed the "
                                       "answer, call arc_submit_answer with it instead of writing the grids; otherwise "
                                       "give the complete final answer again."}


def answer_problems(text: str, answer: list) -> list[str]:
    parts = re.split(TEST_OUTPUT_LABEL, text)
    labelled = {int(label): body for label, body in zip(parts[1::2], parts[2::2])}
    problems = []
    for number, grid in enumerate(answer, 1):
        if grid is None:
            found = [problem for problem in map(grid_problem, json_lists(labelled.get(number, "") if labelled else text)) if problem]
            problems.append(f"output {number}: {found[-1] if found else 'no grid in JSON form, such as [[0, 1], [2, 3]], was found'}")
    return problems


def tool_retry(error: str) -> dict:
    return {"role": "user", "content": f"Your tool call was not run ({error}). The tools are arc_python_executor, with the "
                                       "parameter code and, optionally, target, and arc_submit_answer, with the parameter "
                                       "outputs. Call one of them that way, or give the final answer."}


def pending_confirmation(turns: list[dict]):
    previous = turns[-1] if turns else {}
    calls = (previous.get("assistant") or {}).get("tool_calls") or []
    if calls and calls[-1]["function"]["name"] == "arc_submit_answer" and previous["submissions"][-1]["confirm"]:
        return json.loads(previous["submissions"][-1]["outputs"])
    return None


def apply_turn(body, task: dict, timeout: float, final: bool = False, turns: list[dict] = (), previous: list | None = None,
                  ndimage: bool = False) -> dict:
    empty = {"assistant": None, "calls": [], "submissions": [], "replies": [], "answer": None, "prediction": None}
    if body is None:
        return {"status": "TOKEN_LIMIT", **empty}
    body = collapse_think_end(body)
    try:
        message = parse_assistant(body)
    except ValueError as error:
        if not final and body.startswith("<think>") and body.count("</think>") == 1 and "<tool_call>" in body:
            reasoning, _, content = body[len("<think>"):].partition("</think>")
            return {"status": "REVISE", **empty, "assistant": {"role": "assistant", "reasoning_content": reasoning.strip(),
                                                                "content": content.strip()}, "replies": [tool_retry(str(error))]}
        return {"status": "INVALID_FORMAT", **empty, "error": str(error)}
    if message.get("tool_calls"):
        session = [session_entry(call) for turn in turns for call in turn.get("calls", [])]
        pending, calls, submissions, replies = pending_confirmation(turns), [], [], []
        for call in message["tool_calls"]:
            arguments = tool_arguments(call)
            if call["function"]["name"] == "arc_submit_answer":
                record, reply = submit_answer(arguments["outputs"], task, timeout, session, pending, final, previous, ndimage)
                submissions.append(record)
                if reply is None:
                    return {"status": "ANSWERED", **empty, "assistant": message, "calls": calls, "submissions": submissions,
                            "answer": record["grids"], "prediction": list(record["grids"]), "sources": record["sources"]}
            elif final:
                continue
            else:
                record, reply = run_python_tool(arguments, task, timeout, session, ndimage)
                calls.append(record)
                session.append(session_entry(record))
            replies.append({"role": "tool", "content": reply})
        if final:
            return {"status": "NO_GRID", **empty, "assistant": message, "submissions": submissions}
        return {"status": "REVISE", **empty, "assistant": message, "calls": calls, "submissions": submissions, "replies": replies}
    text = re.sub(r"```python.*?```", "", message["content"], flags=re.DOTALL | re.IGNORECASE)
    answer = answer_grids(text, len(task["test"]))
    if not final and not all(grid is not None for grid in answer):
        last = turns[-1] if turns else None
        if last and last["status"] == "REVISE" and last["answer"] is not None and last["assistant"]["content"] == message["content"]:
            outputs = last_passing(turns)
            return {"status": "NO_GRID", **empty, "assistant": message, "answer": answer,
                    "prediction": list(answer) if outputs is None else outputs, **({} if outputs is None else {"fallback": "program"})}
        return {"status": "REVISE", **empty, "assistant": message, "replies": [answer_retry(answer, answer_problems(text, answer))],
                "answer": answer, "prediction": list(answer)}
    return {"status": "ANSWERED" if any(grid is not None for grid in answer) else "NO_GRID", **empty, "assistant": message,
            "answer": answer, "prediction": list(answer)}


def trajectory(job: dict, task: dict, root: Path, model: str, settings: dict, tokenizer, client) -> dict:
    directory = root / job["key"]
    directory.mkdir()
    ndimage = bool(settings.get("scipy_ndimage"))
    messages = initial_messages(task, settings.get("previous_answer"), settings.get("previous_program"), ndimage)
    result = {**job, "status": "RUNNING", "prediction": None, "turns": 0, "thinking_tokens": 0}
    started = time.monotonic()
    turns, continuation = [], None
    try:
        for turn_index in range(settings["max_turns"]):
            # a turn that starts after deadline_brake_at is the final turn, and so is a turn whose turn_clamp leaves room only for the answer
            now = time.time()
            cap = turn_clamp(settings, now)
            force = now >= settings.get("deadline_brake_at", float("inf")) or (cap is not None and cap <= answer_room(settings))
            final, prefill = start_turn(settings, task, tokenizer, messages, turn_index, turns, force)
            if force:
                result.setdefault("deadline_brake", turn_index)
            try:
                payload = request_payload(tokenizer, messages, model, settings, job["seed"] + turn_index, prefill, continuation, cap)
            except TokenBudgetSpent:
                result["status"] = "TOKEN_LIMIT"
                break
            write_json(directory / f"{turn_index:02d}_request.json", payload)
            response = client(payload, settings["request_timeout"])
            write_json(directory / f"{turn_index:02d}_response.json", response)
            sampled = payload["prompt"] + response["choices"][0]["token_ids"]
            continuation = (decode_tokens(tokenizer, sampled), sampled)
            body = validate_response(tokenizer, payload["prompt"], response, payload["max_tokens"], payload["thinking_token_budget"])
            if prefill:
                body = absorb_prefill(body, prefill)
            turn = apply_turn(body, task, settings["execution_timeout"], final, turns, settings.get("previous_answer"), ndimage)
            write_json(directory / f"{turn_index:02d}_execution.json", turn)
            turns.append(turn)
            result.update(turns=turn_index + 1, prediction=turn["prediction"], status=turn["status"],
                          thinking_tokens=result["thinking_tokens"] + (0 if prefill else reasoning_tokens(tokenizer, response)))
            if final:
                result["forced"] = True
            if turn["status"] != "REVISE" or final:
                break
            messages.extend([turn["assistant"], *turn["replies"]])
        if result["status"] == "REVISE":
            result["status"] = "TURN_LIMIT"
        result.update({key: turns[-1][key] for key in ("sources", "fallback") if turns and key in turns[-1]})
        if not any(grid is not None for grid in result["prediction"] or []):
            fallback = program_fallback(turns)
            if fallback is not None:
                result.update(prediction=fallback, fallback="program")
        programs = program_predictions(turns, result["prediction"])
        if programs is not None:
            result["program_predictions"] = programs
        result["verified"] = verified(turns, result["prediction"])
    except Exception as failure:
        result.update(status="ERROR", error=f"{type(failure).__name__}: {failure}")
    result["elapsed_seconds"] = time.monotonic() - started
    write_json(directory / "result.json", result)
    with RESULT_LOG_LOCK:
        print(json.dumps({"model": model, **result}), flush=True)
    return result

## Scheduler

In [ ]:
import statistics

def atomic_json(path: Path, value) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    temporary.replace(path)


def validate_submission(tasks: dict[str, Any], result: dict[str, Any]) -> None:
    if not tasks or set(result) != set(tasks):
        raise ValueError("submission task coverage differs from challenges")
    for task_id, task in tasks.items():
        if len(result[task_id]) != len(task["test"]):
            raise ValueError("submission test output count differs from challenges")
        for entry in result[task_id]:
            if set(entry) != {"attempt_1", "attempt_2"}:
                raise ValueError("submission requires exactly two attempts")
            for grid in entry.values():
                if normalize_grid(grid) is None or any(type(cell) is not int for row in grid for cell in row):
                    raise ValueError("submission contains an invalid integer grid")


SAMPLING = ("temperature", "top_p", "top_k", "thinking_token_budget", "max_tokens", "max_model_len")
FREE = ("max_turns", "execution_timeout", "deadline_brake", "turn_clamp", "scipy_ndimage")
SEED_OFFSET = 10_000_000


def task_jobs(tasks: dict, seed: int, attempts: int) -> list[dict]:
    jobs = []
    for attempt in range(attempts):
        for task_id in tasks:
            jobs.append({"key": f"d{len(jobs):04d}", "task_id": task_id, "attempt": attempt,
                         "seed": seed + SEED_OFFSET + len(jobs)})
    return jobs


CAPPED, UNVERIFIED, RUNNING, VERIFIED = range(4)  # JobQueue's order of later attempts, by how the first attempt ended


def previous_answer(job: dict, first_keys: dict, results: Path) -> list | None:
    key = first_keys.get(job["task_id"])
    path = results / f"{key}.json"
    grids = json.loads(path.read_text()).get("predictions") if key not in (None, job["key"]) and path.exists() else None
    return grids if grids and any(grid is not None for grid in grids) else None


def earlier_program(job: dict, first_keys: dict, trajectories: Path, task: dict) -> dict | None:
    key = first_keys.get(job["task_id"])
    directory = trajectories / key if key not in (None, job["key"]) else None
    return previous_program(directory, task) if directory is not None and directory.is_dir() else None


class JobQueue:

    def __init__(self, jobs: list[dict], tasks: dict):
        self.verified_count = {}
        self.first = [job for job in jobs if job["attempt"] == 0]
        self.later = [job for job in jobs if job["attempt"] > 0 and not job.get("extra")]
        self.extra = [job for job in jobs if job.get("extra")]
        self.inflight, self.extra_taken, self.rank, self.rates, self.taken = {}, 0, {}, [], 0
        self.lock = threading.Lock()
        self.tasks = tasks

    def record(self, job: dict, result: dict) -> None:
        with self.lock:
            if result.get("completion_tokens") and result.get("elapsed_seconds"):
                self.rates.append(result["completion_tokens"] / result["elapsed_seconds"])
            verified = result.get("verified") and not broken_invariants(self.tasks[job["task_id"]], result.get("predictions"))
            if verified:
                self.verified_count[job["task_id"]] = self.verified_count.get(job["task_id"], 0) + 1
            if job["attempt"] == 0:
                self.rank[job["task_id"]] = (VERIFIED if verified else UNVERIFIED) if result["natural"] else CAPPED

    def rate(self) -> float | None:
        with self.lock:
            return statistics.median(self.rates[-32:]) if len(self.rates) >= 8 else None

    def defer(self, job: dict) -> None:
        with self.lock:
            (self.extra if job.get("extra") else self.later).append(job)
            self.taken -= 1
            self.extra_taken -= bool(job.get("extra"))
            self._finish(job)

    def finish(self, job: dict) -> None:
        with self.lock:
            self._finish(job)

    def _finish(self, job: dict) -> None:
        self.inflight[job["task_id"]] = self.inflight.get(job["task_id"], 1) - 1

    def take(self) -> dict | None:
        with self.lock:
            if self.first:
                job = self.first.pop(0)
            elif self.later:
                job = min(self.later, key=lambda job: self.rank.get(job["task_id"], RUNNING))
                self.later.remove(job)
            else:
                self.extra = [job for job in self.extra if self.verified_count.get(job["task_id"], 0) < 2]
                if not self.extra:
                    return None
                # an output with no job in flight first, then the one with the fewest verified attempts
                job = min(self.extra, key=lambda job: (self.inflight.get(job["task_id"], 0) > 0, self.verified_count.get(job["task_id"], 0)))
                self.extra.remove(job)
                self.extra_taken += 1
            self.inflight[job["task_id"]] = self.inflight.get(job["task_id"], 0) + 1
            self.taken += 1
            return job


def deadline_budget(rate: float | None, remaining: float, budget: int, answer_room: int, minimum: int) -> int | None:
    if rate is None:
        return budget
    fits = int(0.9 * rate * remaining) - answer_room
    return budget if fits >= budget else fits if fits >= minimum else None


def vote_order(rows: list[dict]) -> list[dict]:
    grids = [json.dumps(row.get("prediction")) for row in rows if row.get("prediction") is not None]
    if len(set(grids)) < 3:
        return rows
    verified, count, first = Counter(), Counter(grids), {}
    for row in rows:
        if row.get("prediction") is not None:
            key = json.dumps(row["prediction"])
            verified[key] += bool(row.get("verified"))
            first[key] = min(first.get(key, row["attempt"]), row["attempt"])
    def rank(row):
        if row.get("prediction") is None:
            return (2, 0, 0, row["attempt"], row["attempt"])
        key = json.dumps(row["prediction"])
        return (0 if verified[key] else 1, -verified[key], -count[key], first[key], row["attempt"])
    return sorted(rows, key=rank)


def assemble(tasks: dict, results: list[dict]) -> dict:
    ordered = sorted(results, key=lambda row: (row["attempt"] > 0, row["attempt"] > 0 and not row.get("verified"), row["attempt"]))
    outputs = {}
    for row in ordered:
        outputs.setdefault((row["task_id"], row["test_index"]), []).append(row)
    ordered = [row for rows in outputs.values() for row in vote_order(rows)]
    grids = {}
    for row in ordered:
        found = grids.setdefault((row["task_id"], row["test_index"]), [])
        for grid in (row.get("prediction"), row.get("program_prediction")):
            if grid is not None and grid not in found:
                found.append(grid)
    for row in sorted(results, key=lambda row: row["attempt"]):
        found = grids[(row["task_id"], row["test_index"])]
        if row.get("fill_prediction") is not None and row["fill_prediction"] not in found:
            found.append(row["fill_prediction"])
    submission = {}
    for task_id, task in tasks.items():
        submission[task_id] = []
        for index in range(len(task["test"])):
            found = grids.get((task_id, index), []) + [[[0]], [[0]]]
            submission[task_id].append({"attempt_1": found[0], "attempt_2": found[1]})
    return submission


def fill_predictions(directory: Path, count: int, prediction: list) -> list:
    found = [[] for _ in range(count)]
    order = 0
    for path in sorted(directory.glob("[0-9][0-9]_execution.json")):
        try:
            turn = json.loads(path.read_text())
        except ValueError:  # a turn whose record is still being written
            break
        for call in turn.get("calls") or []:
            for run in [call, *(call.get("functions") or {}).values()]:
                for index, grid in enumerate((run.get("test_outputs") or [])[:count]):
                    if grid is not None:
                        found[index].append((run.get("train_passed") or 0, order, grid))
                order += 1
    mine = list(prediction or []) + [None] * count
    return [next((grid for _, _, grid in sorted(options, key=lambda item: (-item[0], -item[1])) if grid != mine[index]), None)
            for index, options in enumerate(found)]


def salvage(directory: Path) -> list | None:
    turns = []
    for path in sorted(directory.glob("[0-9][0-9]_execution.json")):
        try:
            turns.append(json.loads(path.read_text()))
        except ValueError:
            break
    return program_fallback(turns)


def free_answer(job: dict, task: dict, root: Path, model: str, settings: dict, tokenizer, client, trajectories: Path) -> dict:
    directory = trajectories / job["key"]
    if directory.exists():
        directory.rename(trajectories / f"{job['key']}.error")
    started = time.monotonic()
    try:
        result = trajectory(job, task, trajectories, model, settings, tokenizer, client)
    except Exception as error:
        result = {**job, "status": "ERROR", "prediction": None, "turns": 0, "thinking_tokens": 0,
                  "error": f"{type(error).__name__}: {error}", "elapsed_seconds": time.monotonic() - started}
    predictions = result["prediction"]
    if not any(grid is not None for grid in predictions or []):
        predictions = salvage(directory)
    responses = directory.glob("[0-9][0-9]_response.json")
    row = {key: value for key, value in result.items() if key != "prediction"}
    row.update(predictions=predictions,
               completion_tokens=sum(json.loads(path.read_text())["usage"]["completion_tokens"] for path in responses),
               natural=(result["status"] == "ANSWERED" and "fallback" not in result
                        and result["turns"] < settings["max_turns"]
                        and result["thinking_tokens"] < settings["thinking_token_budget"] - FINAL_ANSWER_RESERVE
                        and not result.get("forced") and "deadline_brake" not in result))
    atomic_json(root / f"{job['key']}.json", row)
    return row


def finalize(tasks: dict, results_dir: Path, destination: Path, jobs: list[dict], trajectories: Path) -> dict:
    # a trajectory still running at the deadline gives the outputs of its programs (salvage)
    results = []
    rows = {path.stem: json.loads(path.read_text()) for path in sorted(results_dir.glob("d[0-9]*[0-9].json"))}
    unfinished = [{**job, "predictions": salvage(trajectories / job["key"]) or []}
                  for job in jobs if job["key"] not in rows and (trajectories / job["key"]).is_dir()]
    for row in [*rows.values(), *unfinished]:  # one result per test output
        programs = row.get("program_predictions") or []
        count = len(tasks[row["task_id"]]["test"])
        fills = []
        if row["attempt"] == 0:
            try:
                fills = fill_predictions(trajectories / row["key"], count, row.get("predictions"))
            except Exception:  # an unreadable record leaves this slot without a fill, and the submission is still written
                fills = []
        results += [
            {**row, "test_index": index, "prediction": grid,
             **({"program_prediction": programs[index]} if index < len(programs) else {}),
             **({"fill_prediction": fills[index]} if index < len(fills) and fills[index] is not None else {})}
            for index, grid in enumerate((row.get("predictions") or []) + [None] * (len(fills) - len(row.get("predictions") or [])))]
    submission = assemble(tasks, results)
    validate_submission(tasks, submission)
    atomic_json(destination, submission)
    return {"answers": len(results), "grids": sum(row.get("prediction") is not None for row in results),
            "outputs_with_grid": sum(entry["attempt_1"] != [[0]] for entries in submission.values() for entry in entries),
            "unfinished": len(unfinished)}

## Engine server

In [ ]:
import ctypes
import signal

def stop_group(process: subprocess.Popen | None) -> None:
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        pass
    # the workers can outlive the leader
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=20)


def mapped_driver(maps: str) -> Path:
    paths = set()
    for line in maps.splitlines():
        fields = line.split(maxsplit=5)
        if len(fields) == 6 and fields[5].startswith('/'):
            path = Path(fields[5])
            if path.name.startswith('libcuda.so.') and path.is_file():
                paths.add(path.resolve())
    if len(paths) != 1:
        raise RuntimeError(f'Expected one loaded CUDA driver, found {len(paths)}')
    driver = paths.pop()
    if 'stubs' in driver.parts:
        raise RuntimeError('The loaded CUDA library is a stub')
    return driver


def link_cuda_driver(directory: Path, environment: dict[str, str]) -> None:
    ctypes.CDLL("libcuda.so.1")  # loads the driver, so that /proc/self/maps names its file
    directory.mkdir(parents=True)
    (directory / "libcuda.so").symlink_to(mapped_driver(Path("/proc/self/maps").read_text()))
    previous = environment.get("LIBRARY_PATH", "")
    environment["LIBRARY_PATH"] = str(directory) + (os.pathsep + previous if previous else "")


# Adapted from halt95's serve-v2.5.sh in the Merlin engine (Apache License 2.0) and modified by us; see THIRD_PARTY_NOTICES.md.
def serve_command(environment: dict) -> tuple[list[str], dict]:
    env = dict(environment)
    assert not (Path.cwd() / "vllm").is_dir(), "start from a directory without a vllm/ beside it (it would shadow the installed engine)"
    cu, cache = SITE / "nvidia/cu13", Path("/kaggle/tmp/vllm-cache-merlin")
    env.update(CUDA_HOME=str(cu), PATH=f"{cu}/bin:{VENV}/bin:{env.get('PATH', '')}")  # for the JIT compilation of FlashInfer and Triton at the first requests
    (cache / "cudalink").mkdir(parents=True, exist_ok=True)  # FlashInfer JIT links with -lcudart, and the CUDA 13 wheel ships only libcudart.so.13
    link = cache / "cudalink/libcudart.so"
    link.unlink(missing_ok=True)
    link.symlink_to(cu / "lib/libcudart.so.13")
    env["LIBRARY_PATH"] = f"{cache}/cudalink:{cu}/lib" + (f":{env['LIBRARY_PATH']}" if env.get("LIBRARY_PATH") else "")
    env.pop("PYTHONPATH", None)
    for name in [name for name in env if name.startswith("MERLIN_")] + ["VLLM_PLE_CPU_OFFLOAD", "VLLM_QSA_KV_SCALES", "VLLM_QSA_KV_COLLECT",
                                                                      "VLLM_QSA_KV_CLIP_COUNT", "VLLM_E2_ALLOW_BREAKABLE", "PYTORCH_CUDA_ALLOC_CONF"]:
        env.pop(name, None)
    flag = Path(env.get("TMPDIR") or "/tmp") / f"arc-ple-loaded.{time.time_ns()}"  # a fresh flag per start: a restart never finds the previous one
    env.update(
        ARC_VLLM_REAP_KEEP=str(FILES / "reap448_keep.json"), VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="0", CUDA_DEVICE_ORDER="PCI_BUS_ID",
        VLLM_SKIP_P2P_CHECK="1", NCCL_P2P_LEVEL="SYS", NCCL_PROTO="LL",  # P2P is on between all four L4s
        VLLM_PP_LAYER_PARTITION="25,23",
        # the PLE offload worker reads the whole checkpoint, which can take over an hour on a network filesystem
        VLLM_E2_PLE_PULL_TRANSPORT="1", VLLM_PLE_OFFLOAD_READY_TIMEOUT="10800", VLLM_PLE_OFFLOAD_HOME_DEVICE="0",
        ARC_VLLM_PLE_LOADED_FLAG=str(flag),
        VLLM_HOST_EMBED_TABLE="1", VLLM_USE_BREAKABLE_CUDAGRAPH="0", VLLM_CACHE_ROOT=str(cache), HF_HUB_OFFLINE="1",
        MERLIN_FULL_K="1", MERLIN_QSA_SORT="1", MERLIN_TIE_DET="1", MERLIN_TIE_RECENT="1", MERLIN_TOPK_SORTED_EMIT="1",
        VLLM_E2_GUARD_MODE="warn", VLLM_E2_GUARD_IDLE="1",
        VLLM_E2_SIDE_CACHE_LAYOUT="1", VLLM_USE_V2_MODEL_RUNNER="1", VLLM_E2_MAMBA_RETAIN_CHECKPOINT="1")
    spec = {"method": "mtp", "model": str(DRAFTER), "num_speculative_tokens": 3, "draft_sample_method": "probabilistic"}
    argv = [str(VENV / "bin/vllm"), "serve", str(MODEL), "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", env["PORT"],
            "--tensor-parallel-size", "2", "--pipeline-parallel-size", "2", "--enable-expert-parallel",
            "--speculative-config", json.dumps(spec, separators=(",", ":")),
            "--max-model-len", env["MAX_MODEL_LEN"], "--max-num-seqs", env["MAX_SEQS"], "--max-num-batched-tokens", "1024",
            "--gpu-memory-utilization", env["UTIL"], "--kv-cache-memory-bytes", env["KV_BYTES"], "--kv-cache-dtype", "fp8_e4m3",
            "--language-model-only",
            "--compilation-config", f'{{"cudagraph_mode":"FULL_AND_PIECEWISE","cudagraph_capture_sizes":[{env["SIZES"]}]}}',
            "--enable-prefix-caching", "--enable-prompt-tokens-details", "--no-enable-flashinfer-autotune",
            "--reasoning-parser", "qwen3", "--shutdown-timeout", "60",
            "--block-size", env["BLOCK_SIZE"], "--use-replayssm", "--prefix-match-unit", "64"]
    if env["INDEXER_KV_DTYPE"]:
        argv += ["--attention-config", f'{{"indexer_kv_dtype":"{env["INDEXER_KV_DTYPE"]}"}}']
    argv += ["--mamba-ssm-cache-dtype", env["SSM_DTYPE"]]
    return argv, env


def launch(environment: dict, stdout) -> subprocess.Popen:
    argv, env = serve_command(environment)
    return subprocess.Popen(argv, env=env, stdout=stdout, stderr=subprocess.STDOUT, start_new_session=True)


def wait_ready(process: subprocess.Popen, config: dict, deadline: float) -> int | None:
    import httpx

    url = f"http://127.0.0.1:{config['port']}/v1/models"
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"the server exited during startup with code {process.returncode}")
        try:
            models = httpx.get(url, timeout=5, trust_env=False).json()["data"]
            if [row["id"] for row in models] == [config["served_model"]]:
                served = models[0].get("max_model_len")
                return int(served) if served else None
        except (httpx.HTTPError, ValueError, KeyError):
            pass
        time.sleep(5)
    raise TimeoutError("vLLM was not ready before the startup deadline")


OUT_OF_MEMORY = ("CUDA out of memory", "OutOfMemoryError", "cudaErrorMemoryAllocation", "CUBLAS_STATUS_ALLOC_FAILED")


def out_of_memory(log_path: Path, tail: int = 1 << 20) -> bool:
    try:
        with log_path.open("rb") as stream:
            stream.seek(max(0, log_path.stat().st_size - tail))
            text = stream.read().decode(errors="replace")
    except OSError:
        return False
    return any(marker in text for marker in OUT_OF_MEMORY)


def server_down(process: subprocess.Popen, config: dict, silence: float = 60.0, pause: float = 5.0) -> bool:
    import httpx

    end = time.time() + silence
    while True:
        if process.poll() is not None:
            return True
        try:
            return httpx.get(f"http://127.0.0.1:{config['port']}/health", timeout=10, trust_env=False).status_code != 200
        except httpx.HTTPError:
            if time.time() >= end:
                return True
        time.sleep(pause)


def generation_metrics(config: dict) -> tuple[float, float] | None:
    import httpx

    try:
        reply = httpx.get(f"http://127.0.0.1:{config['port']}/metrics", timeout=10, trust_env=False)
        reply.raise_for_status()
    except httpx.HTTPError:
        return None
    total = {"vllm:generation_tokens_total": 0.0, "vllm:num_requests_running": 0.0}
    for line in reply.text.splitlines():
        name = line.split("{", 1)[0].split(" ", 1)[0]
        if name in total:
            total[name] += float(line.rsplit(" ", 1)[1])
    return total["vllm:generation_tokens_total"], total["vllm:num_requests_running"]


def probe_generation(config: dict) -> None:
    import httpx

    url = f"http://127.0.0.1:{config['port']}"
    reply = httpx.post(url + "/v1/completions", timeout=900, trust_env=False,
                       json={"model": config["served_model"], "prompt": "1 + 1 =", "max_tokens": 16, "temperature": 0})
    reply.raise_for_status()
    text = (reply.json().get("choices") or [{}])[0].get("text")
    if not text:
        raise RuntimeError("the probe generation returned no text")

## Main Running

In [ ]:
from concurrent.futures import wait
from datetime import datetime, timezone
from functools import partial

def run(config: dict) -> dict:
    import httpx

    started = time.time()
    output = Path(config["output"])
    output.mkdir(parents=True, exist_ok=False)
    (output / "results").mkdir()
    tasks = json.loads(Path(config["challenges"]).read_text())
    if any("output" in test for task in tasks.values() for test in task["test"]):
        raise ValueError("the challenges must not contain test outputs")
    trajectories = Path(config["trajectories"])
    trajectories.mkdir(parents=True)
    settings = config["evaluation"]
    jobs = task_jobs(tasks, settings["seed"], settings["attempts"] + settings["extra_attempts"])
    for job in jobs:
        if job["attempt"] >= settings["attempts"]:
            job["extra"] = True
    queue = JobQueue(jobs, tasks)
    destination = Path(config["final_submission"])
    generation_deadline = config["deadline"] - config["cleanup_reserve"]
    state = {"status": "STARTING", "started_utc": datetime.now(timezone.utc).isoformat(), "jobs": len(jobs)}
    atomic_json(output / "status.json", state)
    environment = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
                   "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "1", **config.get("env", {})}
    server, log, pool, stopping = None, None, None, threading.Event()
    try:
        link_cuda_driver(output / "cuda-driver-link", environment)  # FlashInfer's JIT builds link with -lcuda
        # fallback_env: the levels tried in turn when the server fails to start or to answer probe_generation, and the level that a restart after running out of GPU memory moves down to (recover)
        log_path = output / "server.log"
        log = log_path.open("x")
        primary_environment = environment  # the ladder's levels replace variables of this one
        server = launch(environment, log)
        try:
            state["served_max_model_len"] = wait_ready(server, config, min(generation_deadline, time.time() + config["startup_timeout"]))
            probe_generation(config)
        except (RuntimeError, TimeoutError, httpx.HTTPError) as error:
            state["fallback"] = {"error": f"{type(error).__name__}: {error}", "seconds": round(time.time() - started, 1)}
            atomic_json(output / "status.json", state)
            stop_group(server)
            log.close()
            levels = config["fallback_env"]
            for level, variables in enumerate(levels, 1):  # a level that exits during startup is followed by the next one
                environment = {**primary_environment, **variables}
                log_path = output / ("server_fallback.log" if level == 1 else f"server_fallback{level}.log")
                log = log_path.open("x")
                server = launch(environment, log)
                try:
                    state["served_max_model_len"] = wait_ready(server, config, generation_deadline)  # no startup time limit: without a server nothing is answered
                    state["fallback"]["level"] = level
                    break
                except RuntimeError as failure:
                    stop_group(server)
                    log.close()
                    state["fallback"].setdefault("failed_levels", []).append(f"{type(failure).__name__}: {failure}"[:300])
                    atomic_json(output / "status.json", state)
                    if level == len(levels):
                        raise
        state.update(status="GENERATING", ready_seconds=round(time.time() - started, 1), restarts=[])
        atomic_json(output / "status.json", state)
        tokenizer = load_tokenizer(config["model"])
        request = {key: settings[key] for key in SAMPLING + FREE}
        client = partial(complete, httpx.Client(trust_env=False, follow_redirects=False), f"http://127.0.0.1:{config['port']}/v1/completions",
                         tokenizer)
        if request["scipy_ndimage"] and (error := ndimage_error()):
            request.pop("scipy_ndimage")
            print(f"scipy.ndimage failed to import, so the prompts leave it out: {error}", flush=True)
        first_keys = {job["task_id"]: job["key"] for job in jobs if job["attempt"] == 0}
        restart_lock, done = threading.Lock(), []

        def recover(seen: int, error: str, stalled: bool = False) -> dict | None:
            nonlocal server, environment, log_path
            with restart_lock:
                if len(done) > seen:
                    return done[seen]
                if stopping.is_set() or not (stalled or server_down(server, config)):
                    return None
                at, restarts = time.time(), state["restarts"]
                reason = "generation stalled" if stalled else "no health answer" if server.poll() is None else f"exit code {server.returncode}"
                record = {"at_seconds": round(at - started, 1), "reason": reason, "error": error[:300]}
                restarts.append(record)  # before the wait: a start still waiting when the run finalizes is in status.json
                try:
                    stop_group(server)
                    for _ in range(120):  # its engine and workers must let go of GPU and host memory first
                        try:
                            os.killpg(server.pid, 0)
                        except ProcessLookupError:
                            break
                        time.sleep(1)
                    restart_log = output / f"server_restart{len(restarts)}.log"
                    levels, level = config["fallback_env"], (state.get("fallback") or {}).get("level", 0)
                    oom, stuck = out_of_memory(log_path), state.get("failed_starts", 0) >= 2
                    if level < len(levels) and (oom or stuck):
                        # a serving that ran out of GPU memory restarts one level down (e.g. a smaller KV pool), and so does one that failed to start twice in a row (a level whose settings cannot start)
                        environment = {**primary_environment, **levels[level]}
                        state["fallback"] = {**(state.get("fallback") or {}),
                                             "error": "the server ran out of GPU memory" if oom else "the serving failed to start twice",
                                             "at_seconds": record["at_seconds"], "level": level + 1}
                        state["failed_starts"] = 0
                        record.update(fallback=True, fallback_level=level + 1)
                    log_path = restart_log
                    threading.Thread(target=prefetch, args=(Path(config["model"]),), daemon=True).start()
                    with restart_log.open("x") as handle:
                        server = launch(environment, handle)
                    if stopping.is_set():  # the run's finally may have stopped the old process only
                        stop_group(server)
                        return None
                    state["served_max_model_len"] = wait_ready(server, config, generation_deadline)
                    record.update(ready_seconds=round(time.time() - at, 1), requeued=[], deferred=[])
                    state["failed_starts"] = 0
                    done.append(record)
                except Exception as failure:  # counts as an attempt; the next ERROR tries again
                    stop_group(server)
                    record["failed"] = f"{type(failure).__name__}: {failure}"
                    state["failed_starts"] = state.get("failed_starts", 0) + 1
                record["wait_seconds"] = round(time.time() - at, 1)
                print(json.dumps({"server_restart": {key: value for key, value in record.items() if key not in ("requeued", "deferred")}}), flush=True)
                return done[seen] if len(done) > seen and not stopping.is_set() else None

        def stall_watch(limit: float) -> None:
            progress, last = time.time(), None
            while not stopping.wait(30.0) and time.time() < generation_deadline:
                seen, metrics = len(done), generation_metrics(config)
                if metrics is not None and (metrics[1] == 0 or metrics[0] != last):
                    progress, last = time.time(), metrics[0]
                elif time.time() - progress >= limit:
                    running = "" if metrics is None else f" with {metrics[1]:.0f} running"
                    recover(seen, f"no generation for {time.time() - progress:.0f} s{running}", stalled=True)
                    progress, last = time.time(), None

        def answer(job):
            status, retry, deaths, aside = "SKIPPED", False, 0, job.pop("aside", None)
            while True:
                if time.time() >= generation_deadline:
                    break
                answer_room = settings["max_tokens"] - settings["thinking_token_budget"]
                budget = deadline_budget(queue.rate(), generation_deadline - time.time(), settings["thinking_token_budget"],
                                         answer_room, settings["deadline_budget"])
                if budget is None:
                    break
                if aside:
                    name, n = aside, 0
                    while aside == "died" and ((output / "results" / f"{job['key']}.{name}.json").exists()
                                               or (trajectories / f"{job['key']}.{name}").exists()):
                        n += 1
                        name = f"died-{chr(97 + n)}"  # no trailing digit: finalize's d[0-9]*[0-9].json must not match
                    (output / "results" / f"{job['key']}.json").rename(output / "results" / f"{job['key']}.{name}.json")
                    if aside == "died" and (trajectories / job["key"]).exists():
                        (trajectories / job["key"]).rename(trajectories / f"{job['key']}.{name}")
                    aside = None
                seen = len(done)
                sampling = {**request, "thinking_token_budget": budget, "max_tokens": budget + answer_room,
                            "request_timeout": max(1.0, generation_deadline - time.time())}
                served = state.get("served_max_model_len")
                if served and sampling.get("max_model_len", 0) > served:
                    # a fallback level may serve a shorter context: no request may ask past it, or the server refuses every one (HTTP 400)
                    sampling.update(max_model_len=served, thinking_token_budget=min(sampling["thinking_token_budget"], served),
                                    max_tokens=min(sampling["max_tokens"], served))
                if sampling.get("deadline_brake"):
                    sampling["deadline_brake_at"] = generation_deadline - sampling.pop("deadline_brake")
                if sampling.get("turn_clamp"):
                    margin, rate = sampling.pop("turn_clamp"), queue.rate()
                    if rate:  # none before 8 finished requests: those early jobs run unclamped
                        sampling.update(turn_clamp_at=generation_deadline - margin, turn_clamp_rate=rate)
                if job["attempt"] > 0 and not job.get("extra"):  # a later attempt sees the first one's answer and program
                    previous = previous_answer(job, first_keys, output / "results")
                    if previous is not None:
                        sampling["previous_answer"] = previous
                        try:
                            program = earlier_program(job, first_keys, trajectories, tasks[job["task_id"]])
                        except Exception:  # an unreadable first attempt: the job runs with the answer alone
                            program = None
                        if program is not None:
                            sampling["previous_program"] = program
                result = free_answer(job, tasks[job["task_id"]], output / "results", config["served_model"], sampling,
                                     tokenizer, client, trajectories)
                status = result["status"]
                restart = recover(seen, result.get("error", "")) if status == "ERROR" else None
                if restart:
                    deaths += 1
                    restart["requeued" if deaths == 1 else "deferred"].append(job["key"])
                    if deaths == 1:
                        aside = "died"
                        continue
                    queue.defer({**job, "aside": "died"})
                    return None
                queue.record(job, result)
                if status != "ERROR" or retry:
                    break
                retry, aside = True, "error"
            return status

        statuses = []

        def worker():
            while time.time() < generation_deadline and (job := queue.take()) is not None:
                if (status := answer(job)) is not None:
                    statuses.append(status)
                    queue.finish(job)

        # one thread per in-flight request; the server's max_num_seqs bounds what runs at once
        pool = ThreadPoolExecutor(max_workers=settings["concurrency"])
        threading.Thread(target=stall_watch, args=(config["stall_restart"],), daemon=True).start()
        futures = [pool.submit(worker) for _ in range(settings["concurrency"])]
        wait(futures, timeout=max(0.0, generation_deadline - time.time()))
        finished = list(statuses)
        state.update(status="FINALIZING", finished=len(finished), abandoned=queue.taken - len(finished),
                     not_started=len(jobs) - queue.taken, status_counts={name: finished.count(name) for name in sorted(set(finished))},
                     extra_started=queue.extra_taken)
    except BaseException as error:
        state.update(status="ERROR", error=f"{type(error).__name__}: {error}")
    finally:
        stopping.set()  # before `server` is read: a restart in progress then stops what it started (recover)
        if pool is not None:
            pool.shutdown(wait=False, cancel_futures=True)
        if server is not None:
            stop_group(server)
        if log is not None:
            log.close()
    # a run without a server or without a single grid raises: Kaggle then shows a failed submission, where placeholders would score 0
    try:
        if "ready_seconds" not in state:
            raise RuntimeError(f"the server was never ready ({state.get('error')})")
        state["finalization"] = finalize(tasks, output / "results", destination, jobs, trajectories)
        if not state["finalization"]["outputs_with_grid"]:
            destination.unlink()
            raise RuntimeError("no output has a grid")
        if state["status"] != "ERROR":
            state["status"] = "COMPLETE"
        return state
    except Exception as error:
        state.update(status="ERROR", error=f"{type(error).__name__}: {error}")
        raise
    finally:
        state["released_utc"] = datetime.now(timezone.utc).isoformat()
        atomic_json(output / "status.json", state)

## Configuration

In [ ]:
FALLBACK = {"BLOCK_SIZE": "4096", "INDEXER_KV_DTYPE": "", "ARC_VLLM_GDN_PARK": "0",
            "CUDA_VISIBLE_DEVICES": "0,1,2,3", "MAX_MODEL_LEN": "131072", "ARC_VLLM_QSA_STAGING_BLOCKS": "0", "ARC_VLLM_QSA_SWAP_OVERLAP": ""}
if SCORING:
    shutil.copyfile(AZ_FOCUS, "/kaggle/tmp/override.safetensors")
    config = {
        "served_model": "qwen",
        "port": 8580,
        "evaluation": {
            "seed": 20260926,
            "attempts": 2,
            "extra_attempts": 3,
            "concurrency": 64,
            "temperature": 0.7,
            "top_p": 0.95,
            "top_k": 20,
            "thinking_token_budget": 196608,
            "max_tokens": 204800,
            "max_model_len": 196608,
            "deadline_budget": 16384,
            "max_turns": 1000,
            "execution_timeout": 5.0,
            "deadline_brake": 900,
            "turn_clamp": 180,
            "scipy_ndimage": True,
        },
        "env": {
            "ARC_VLLM_DRAFT_VOCAB": str(FILES / "qwen38_arc_16384.json"),
            "ARC_VLLM_WEIGHT_OVERRIDE": "/kaggle/tmp/override.safetensors",
            "ARC_VLLM_GDN_REUSE": "1",
            "KV_BYTES": "3900000000",
            "UTIL": "0.95",
            "MAX_SEQS": "64",
            "PORT": "8580",
            "SIZES": "1,2,3,4,6,8,12,16,24,32,48,64,96,128,144,160,176,192,208,224,240,256",
            "ARC_VLLM_TOPK_LOGITS": "20",
            "ARC_VLLM_RETAIN_LAST_STATE": "1",
            "ARC_VLLM_RETAIN_LAST_STATE_PROTECT_SECONDS": "300",
            "ARC_VLLM_QSA_HOST_KV": "1",
            "ARC_VLLM_QSA_HOST_KV_SIZE": str(19 * 10**9),
            "ARC_VLLM_READ_FILTER": "1",
            "ARC_VLLM_LATE_COMMIT": "1",
            "SSM_DTYPE": "bfloat16",
            "ARC_VLLM_GDN_PACK": "tight",
            "ARC_VLLM_QSA_DEVICE_STEPS": "1",
            "ARC_VLLM_QSA_DEVICE_TOKENS": "5344",
            "BLOCK_SIZE": "8192",
            "INDEXER_KV_DTYPE": "fp8",
            "ARC_VLLM_GDN_PARK": "2048",
            "ARC_VLLM_QSA_SWAP_OVERLAP": "1",
            "CUDA_DEVICE_MAX_CONNECTIONS": "32",
            "CUDA_VISIBLE_DEVICES": "0,2,1,3",
            "MAX_MODEL_LEN": "196608",
            "ARC_VLLM_QSA_STAGING_BLOCKS": "34",
            "ARC_VLLM_VERIFY_CHUNK_SIZE": str(32 << 20),
        },
        "fallback_env": [
            {**FALLBACK, "MAX_SEQS": "48", "KV_BYTES": "4080000000", "SIZES": "1,2,3,4,6,8,12,16,24,32,48,64,96,128,144,160,176,192",
             "ARC_VLLM_QSA_DEVICE_TOKENS": "3088", "ARC_VLLM_QSA_HOST_KV_SIZE": str(12 * 10**9)},
            {**FALLBACK, "MAX_SEQS": "36", "KV_BYTES": "4100000000", "SIZES": "1,2,3,4,6,8,12,16,24,32,48,64,96,128",
             "ARC_VLLM_QSA_DEVICE_STEPS": "0", "ARC_VLLM_QSA_DEVICE_TOKENS": "0", "ARC_VLLM_GDN_PACK": "1", "ARC_VLLM_QSA_HOST_KV_SIZE": str(10 * 10**9)},
            {**FALLBACK, "MAX_SEQS": "20", "KV_BYTES": "4170000000", "SIZES": "1,2,3,4,6,8,12,16,24,32,48,64,80",
             "ARC_VLLM_QSA_DEVICE_STEPS": "0", "ARC_VLLM_QSA_DEVICE_TOKENS": "0", "ARC_VLLM_GDN_PACK": "1", "ARC_VLLM_QSA_HOST_KV_SIZE": str(10 * 10**9)},
        ],
        "startup_timeout": 10800,
        "cleanup_reserve": 300,
        "stall_restart": 600,
        "model": str(MODEL),
        "challenges": str(CHALLENGES),
        "trajectories": "/kaggle/tmp/trajectories",
        "deadline": STARTED + 42300,
        "output": "/kaggle/working/run",
        "final_submission": "/kaggle/working/submission.json",
    }

In [ ]:
if SCORING:
    state = run(config)
    print(json.dumps(state, indent=2)[:20000], flush=True)
# Kaggle can submit a saved notebook version only if it wrote `submission.json`, so a saved run writes a placeholder.
else:
    Path("/kaggle/working/submission.json").write_text(json.dumps(
        {task_id: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in task["test"]] for task_id, task in TASKS.items()}))